### Milestone 2 – Email Assistant

### PART 1: Dataset Preparation
Added new columns ideal_intent and ideal_tone in sample_emails_with_triage_200 dataset

### PART 2: Evaluation Notebook

In [1]:
import pandas as pd

Created notebook and loaded the dataset

In [2]:
df = pd.read_csv("../data/sample_emails_with_triage_200.csv")
df.head()

,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,respond,polite
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,respond,polite
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,respond,polite
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,respond,polite
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,respond,polite


### PART 3: Email Assistant Logic (Reuse from Milestone 1)

In [3]:
def email_assistant(email_text):
    text = str(email_text).lower()

    if any(k in text for k in ["urgent", "deadline", "submit", "due", "payment", "invoice", "overdue"]):
        return "respond", "urgent"
    if any(k in text for k in ["suspicious", "security alert", "account suspended", "login attempts"]):
        return "notify", "urgent"
    if any(k in text for k in ["sale", "offer", "discount", "congratulations", "winner"]):
        return "ignore", "neutral"
    return "respond", "neutral"

### PART 4: Generate Predictions

In [4]:
predictions = []

for _, row in df.iterrows():
    action, tone = email_assistant(row["body"])
    predictions.append({
        "id": row["id"],
        "predicted_intent": action,
        "predicted_tone": tone,
    })

pred_df = pd.DataFrame(predictions)
pred_df.head()

,id,predicted_intent,predicted_tone
0,1,respond,neutral
1,2,respond,urgent
2,3,respond,neutral
3,4,respond,neutral
4,5,respond,neutral


### PART 5: Evaluate Accuracy

In [6]:
def evaluate(row):
    score = 0
    if row["predicted_intent"] == row["ideal_intent"]:
        score += 1
    if row["predicted_tone"] == row["ideal_tone"]:
        score += 1
    return score

eval_df = df.merge(pred_df, on="id")
eval_df["score"] = eval_df.apply(evaluate, axis=1)

accuracy = (eval_df["score"].sum() / (len(eval_df) * 2)) * 100
print("Overall accuracy (%):", accuracy)

Overall accuracy (%): 56.00000000000001


### PART 6: Save Evaluation Output

In [7]:
out_path = "../data/milestone2_output_isha-bhole.csv"
eval_df.to_csv(out_path, index=False)
print("Saved:", out_path)

Saved: ../data/milestone2_output_isha-bhole.csv
